# CodeGen — Group 45
## Step 3b: Python→Rust **translation** (Python given at the input)

This is the **translation** variant of Step 3. In Step 3 (generation) the model only saw a Rust
signature + English description and had to invent the whole solution. Here we put the **known
Python solution into the input** and ask the model to produce the equivalent Rust. The algorithm is
handed over, so the 350M model spends its capacity on Rust syntax — a fairer, easier job, and it is
the task the project is named after.

We report this **"translation" number next to the "generation" number** — not as a replacement.

**Pipeline:** Rust-aware base (Step 2.5) → LoRA fine-tune on `pairs_v2.jsonl` **with Python in the
input** → evaluate on **HumanEval-Rust** with each problem's **Python** as input.

**Leakage:** still train on MBPP, test on HumanEval — disjoint. The Python fed at eval is the
HumanEval *Python source*, never the Rust answer. Run on a **T4 GPU**, then `Runtime → Run all`.

## 1. Install Rust + libraries
We do **not** upgrade `torch` (breaks Colab's torchvision). Add `peft`; remove old `torchao`.

In [1]:
!curl https://sh.rustup.rs -sSf | sh -s -- -y -q
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
!rustc --version
# IMPORTANT: do NOT add `torch` here.
!pip install -q -U datasets transformers accelerate peft
!pip uninstall -q -y torchao
print("setup done")

warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.

  stable-x86_64-unknown-linux-gnu installed - rustc 1.96.1 (31fca3adb 2026-06-26)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leading DOT):
. "$HOME/.cargo/env"            # For sh/bash/zsh/ash/dash/pdksh
source "$HOME/.cargo/env.fish"  # For fish
source "~/.cargo/env.nu"  # For nushell
source "$HOME/.cargo/env.tcsh"  # For tcsh
. "$HOME/.cargo/env.ps1"        # For pwsh
source "$HOME/.cargo/env.xsh"   # For xonsh
rustc 1.96.1 (31fca3

## 2. Mount Drive, load the Rust-aware base + pairs_v3.jsonl
We start from the monolingual Rust-aware base (Step 2.5) and train on the larger multi-sampled
`pairs_v3.jsonl`. Both live in your Drive folder.

In [2]:
import os
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/CodeGen_Group45"
BASE  = DRIVE + "/codegen350m-rust-base"     # Rust-aware base from Step 2.5
PAIRS = "pairs_v3.jsonl"

import shutil
if not os.path.exists(PAIRS):
    shutil.copy(DRIVE + "/pairs_v3.jsonl", PAIRS)

from datasets import load_dataset
data = load_dataset("json", data_files=PAIRS, split="train")
assert os.path.isdir(BASE), "Rust-aware base not found: " + BASE
print(len(data), "pairs loaded from", PAIRS, "| base:", BASE)

Mounted at /content/drive


Generating train split: 0 examples [00:00, ? examples/s]

643 pairs loaded from pairs_v3.jsonl | base: /content/drive/MyDrive/CodeGen_Group45/codegen350m-rust-base


## 3. Load the base model and attach LoRA
Fresh LoRA adapter on the Rust-aware base (the generation adapter was a different objective).

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE)
model.config.pad_token_id = tok.eos_token_id

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                  task_type="CAUSAL_LM", target_modules=["qkv_proj", "out_proj"])
model = get_peft_model(model, lora)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.print_trainable_parameters()

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

trainable params: 1,966,080 || all params: 358,678,528 || trainable%: 0.5481


## 4. Train with the Python in the input (translation)
Only change from Step 3 generation: the input is now **Python (as a comment) + the Rust
signature**. We still mask everything before the body, so the loss is only on the Rust we want.
The model learns: *given the Python and the Rust signature, write the Rust body.*

In [4]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

def python_as_comment(py):
    if not py:
        return ""
    body = "\n".join("// " + line for line in py.strip().splitlines())
    return "// Reference Python implementation:\n" + body + "\n"

MAXLEN = 1024   # larger than Step 3 (512): the Python adds input length

def build_example(ex):
    # INPUT = Python comment + Rust signature ; TARGET = Rust body (after the signature)
    prompt_text = python_as_comment(ex["python"]) + ex["rust_prompt"]
    full_text   = python_as_comment(ex["python"]) + ex["rust_solution"] + tok.eos_token
    full_ids    = tok(full_text,   truncation=True, max_length=MAXLEN)["input_ids"]
    prompt_ids  = tok(prompt_text, truncation=True, max_length=MAXLEN)["input_ids"]
    k = min(len(prompt_ids), len(full_ids))
    labels = [-100]*k + full_ids[k:]            # train ONLY on the Rust body
    return {"input_ids": full_ids, "attention_mask": [1]*len(full_ids), "labels": labels}

tok_ds   = data.map(build_example, remove_columns=data.column_names)
collator = DataCollatorForSeq2Seq(tok, model=model, label_pad_token_id=-100, padding=True)

args = TrainingArguments(output_dir="ckpt_translate", per_device_train_batch_size=4,
    gradient_accumulation_steps=2, num_train_epochs=3, learning_rate=1e-4,
    fp16=torch.cuda.is_available(), logging_steps=20, save_strategy="no", report_to="none")
Trainer(model=model, args=args, train_dataset=tok_ds, data_collator=collator).train()
print("\ntranslation fine-tune done")

Map:   0%|          | 0/643 [00:00<?, ? examples/s]

Step,Training Loss
20,0.687232
40,0.607195
60,0.545150
80,0.509205
100,0.481041
120,0.477027
140,0.486877
160,0.386906
180,0.388167
200,0.401616



translation fine-tune done


## 5. Save the translation adapter

In [5]:
model.save_pretrained("codegen350m-rust-lora-translate")
import shutil
shutil.copytree("codegen350m-rust-lora-translate",
                DRIVE + "/codegen350m-rust-lora-translate", dirs_exist_ok=True)
print("adapter saved (local + Drive)")

adapter saved (local + Drive)


## 6. Evaluate translation on HumanEval-Rust
Feed each HumanEval problem's **Python** solution as input; the model produces Rust; the harness
compiles + tests it. We join the Rust problems (`humaneval-rs`) to their Python (`openai_humaneval`)
by id. The **fixed `trim_to_body`** now ignores braces inside strings/char-literals/comments, so
`println!("}")` no longer truncates the body early.

In [6]:
import re, subprocess, tempfile, os
from collections import Counter

def evaluate_one(prompt, completion, tests, compile_timeout=60, run_timeout=10):
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src, binp = os.path.join(wd, "main.rs"), os.path.join(wd, "prog")
        open(src, "w").write(program)
        try:
            c = subprocess.run(["rustc", src, "-o", binp], capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "compile_timeout"
        if c.returncode != 0:
            return "compile_error"
        try:
            r = subprocess.run([binp], capture_output=True, text=True, timeout=run_timeout)
        except subprocess.TimeoutExpired:
            return "run_timeout"
        return "pass" if r.returncode == 0 else "run_fail"

def trim_to_body(text):
    # Cut at the brace that closes the function, IGNORING braces inside strings/chars/comments.
    depth = 1
    i, n = 0, len(text)
    in_str = in_char = in_line = in_block = False
    while i < n:
        ch = text[i]
        nxt = text[i+1] if i+1 < n else ""
        if in_line:
            if ch == "\n": in_line = False
            i += 1; continue
        if in_block:
            if ch == "*" and nxt == "/": in_block = False; i += 2; continue
            i += 1; continue
        if in_str:
            if ch == "\\": i += 2; continue
            if ch == '"': in_str = False
            i += 1; continue
        if in_char:
            if ch == "\\": i += 2; continue
            if ch == "'": in_char = False
            i += 1; continue
        if ch == "/" and nxt == "/": in_line = True; i += 2; continue
        if ch == "/" and nxt == "*": in_block = True; i += 2; continue
        if ch == '"': in_str = True; i += 1; continue
        if ch == "'":
            if nxt == "\\" or (i+2 < n and text[i+2] == "'"): in_char = True
            i += 1; continue
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return text[:i]
        i += 1
    return text

def load_rs(cfg):
    try:    return load_dataset("nuprl/MultiPL-E", cfg, split="test")
    except Exception: return load_dataset("nuprl/MultiPL-E", cfg, split="test", trust_remote_code=True)

eval_ds = load_rs("humaneval-rs")

# Python source for each HumanEval problem (the input side of translation)
pyset = load_dataset("openai/openai_humaneval", split="test")
py_by_id = {}
for ex in pyset:
    num = int(ex["task_id"].split("/")[1])      # "HumanEval/0" -> 0
    py_by_id[num] = ex["prompt"] + ex["canonical_solution"]

def rs_id(name):
    m = re.search(r"HumanEval_(\d+)_", name)
    return int(m.group(1)) if m else None

model.eval()
def model_completion(ex, max_new_tokens=512):
    pid = rs_id(ex["name"])
    pyc = python_as_comment(py_by_id.get(pid, ""))   # Python in the input (translation)
    prompt = pyc + ex["prompt"]
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return trim_to_body(text)

statuses, matched = [], 0
for ex in eval_ds:
    if rs_id(ex["name"]) in py_by_id: matched += 1
    statuses.append(evaluate_one(ex["prompt"], model_completion(ex), ex["tests"]))

n_pass = sum(s == "pass" for s in statuses)
acc = 100 * n_pass / len(statuses)
print("="*54)
print(f"Python->Rust TRANSLATION on HumanEval-Rust  ({matched}/{len(eval_ds)} had Python)")
print(f"  execution accuracy (pass): {acc:.1f}%   ({n_pass}/{len(statuses)})")
print(f"  breakdown: {dict(Counter(statuses))}")
print(f"  reference: generation pass was ~1.9%")
print("="*54)

README.md:   0%|          | 0.00/33.2k [00:00<?, ?B/s]

humaneval-rs/test-00000-of-00001.parquet:   0%|          | 0.00/75.3k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/156 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Python->Rust TRANSLATION on HumanEval-Rust  (156/156 had Python)
  execution accuracy (pass): 7.1%   (11/156)
  breakdown: {'compile_error': 118, 'pass': 11, 'run_fail': 27}
  reference: generation pass was ~1.9%


## What this gives you
- A **translation** number (Python→Rust) to report **next to** the generation number — both honest,
  both on HumanEval-Rust, clearly labelled. This is the task the proposal names.
- The fixed `trim_to_body` (string/comment-aware) also helps the generation Step 3 and the Step 2
  data engine — worth copying into both.

**Next (CP3):** add the **RAG** layer — retrieve the top-K most similar Python→Rust examples and
prepend them on top of this translation model.